# Feature Engineering

The objective of this notebook is to transform the raw demand data into a machine learning-ready dataset.

Based on the insights obtained during Exploratory Data Analysis (EDA), we will engineer temporal, historical, promotional, and categorical features that help the forecasting model learn demand patterns.

The final output of this notebook will be a processed dataset that will be used for model training.

In [63]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [64]:
demand_df = pd.read_csv("../data/raw/demand.csv")
promo_df = pd.read_csv("../data/raw/promotions.csv")

In [65]:
demand_df["date"] = pd.to_datetime(demand_df["date"])

promo_df["promotion_date"] = pd.to_datetime(
    promo_df["promotion_date"]
)

In [66]:
print(demand_df.shape)
print(promo_df.shape)

demand_df.head()

(9855, 4)
(15, 4)


,date,demand,sku,supermarket
0,2019-01-01,93.0,Organic Milk,FreshMart
1,2019-01-02,93.0,Organic Milk,FreshMart
2,2019-01-03,94.0,Organic Milk,FreshMart
3,2019-01-04,95.0,Organic Milk,FreshMart
4,2019-01-05,92.0,Organic Milk,FreshMart


In [67]:
demand_df["demand"] = (
    demand_df
    .groupby(["supermarket", "sku"])["demand"]
    .transform(lambda x: x.fillna(x.median()))
)

In [68]:
print(demand_df["demand"].isnull().sum())

0


In [69]:
promo_df = promo_df.drop(columns=["Unnamed: 0"])

In [70]:
promo_df["promotion_end"] = (
    promo_df["promotion_date"] +
    pd.Timedelta(days=6)
)

In [71]:
demand_df["promotion"] = 0

In [72]:
for _, row in promo_df.iterrows():

    mask = (
        (demand_df["date"] >= row["promotion_date"]) &
        (demand_df["date"] <= row["promotion_end"]) &
        (demand_df["sku"] == row["sku"]) &
        (demand_df["supermarket"] == row["supermarket"])
    )

    demand_df.loc[mask, "promotion"] = 1

In [73]:
print(demand_df["promotion"].value_counts())

promotion
0    9750
1     105
Name: count, dtype: int64


In [74]:
# Year
demand_df["year"] = demand_df["date"].dt.year

# Month (numeric: 1-12)
demand_df["month"] = demand_df["date"].dt.month

# Quarter (1-4)
demand_df["quarter"] = demand_df["date"].dt.quarter

# Week of year
demand_df["week_of_year"] = demand_df["date"].dt.isocalendar().week.astype(int)

# Day of week (0=Monday, 6=Sunday)
demand_df["day_of_week"] = demand_df["date"].dt.dayofweek

# Day of month
demand_df["day_of_month"] = demand_df["date"].dt.day

# Weekend indicator
demand_df["is_weekend"] = (
    demand_df["day_of_week"] >= 5
).astype(int)

In [75]:
demand_df.head()

,date,demand,sku,supermarket,promotion,year,month,quarter,week_of_year,day_of_week,day_of_month,is_weekend
0,2019-01-01,93.0,Organic Milk,FreshMart,0,2019,1,1,1,1,1,0
1,2019-01-02,93.0,Organic Milk,FreshMart,0,2019,1,1,1,2,2,0
2,2019-01-03,94.0,Organic Milk,FreshMart,0,2019,1,1,1,3,3,0
3,2019-01-04,95.0,Organic Milk,FreshMart,0,2019,1,1,1,4,4,0
4,2019-01-05,92.0,Organic Milk,FreshMart,0,2019,1,1,1,5,5,1


In [76]:
demand_df = demand_df.sort_values(
    by=["supermarket", "sku", "date"]
).reset_index(drop=True)

In [77]:
lag_days = [1, 7, 14, 28, 56]

for lag in lag_days:
    demand_df[f"lag_{lag}"] = (
        demand_df
        .groupby(["supermarket", "sku"])["demand"]
        .shift(lag)
    )

In [78]:
demand_df[
    [
        "date",
        "supermarket",
        "sku",
        "demand",
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "lag_56"
    ]
].head(60)

,date,supermarket,sku,demand,lag_1,lag_7,lag_14,lag_28,lag_56
0,2019-01-01,DailyNeeds,Free Range Eggs,68.0,NaN,NaN,NaN,NaN,NaN
1,2019-01-02,DailyNeeds,Free Range Eggs,68.0,68.0,NaN,NaN,NaN,NaN
2,2019-01-03,DailyNeeds,Free Range Eggs,68.0,68.0,NaN,NaN,NaN,NaN
3,2019-01-04,DailyNeeds,Free Range Eggs,68.0,68.0,NaN,NaN,NaN,NaN
4,2019-01-05,DailyNeeds,Free Range Eggs,71.0,68.0,NaN,NaN,NaN,NaN
5,2019-01-06,DailyNeeds,Free Range Eggs,65.0,71.0,NaN,NaN,NaN,NaN
6,2019-01-07,DailyNeeds,Free Range Eggs,65.0,65.0,NaN,NaN,NaN,NaN
7,2019-01-08,DailyNeeds,Free Range Eggs,65.0,65.0,68.0,NaN,NaN,NaN
8,2019-01-09,DailyNeeds,Free Range Eggs,69.0,65.0,68.0,NaN,NaN,NaN
9,2019-01-10,DailyNeeds,Free Range Eggs,69.0,69.0,68.0,NaN,NaN,NaN


In [79]:
demand_df["demand_change_1"] = (
    demand_df["demand"] -
    demand_df["lag_1"]
)

demand_df["demand_change_7"] = (
    demand_df["demand"] -
    demand_df["lag_7"]
)

In [80]:
rolling_windows = [7, 14, 28]

for window in rolling_windows:

    demand_df[f"rolling_mean_{window}"] = (
        demand_df
        .groupby(["supermarket", "sku"])["demand"]
        .transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

    demand_df[f"rolling_std_{window}"] = (
        demand_df
        .groupby(["supermarket", "sku"])["demand"]
        .transform(
            lambda x: x.shift(1).rolling(window).std()
        )
    )

In [81]:
demand_df[
    [
        "date",
        "demand",
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28"
    ]
].head(40)

,date,demand,rolling_mean_7,rolling_mean_14,rolling_mean_28,rolling_std_7,rolling_std_28
0,2019-01-01,68.0,NaN,NaN,NaN,NaN,NaN
1,2019-01-02,68.0,NaN,NaN,NaN,NaN,NaN
2,2019-01-03,68.0,NaN,NaN,NaN,NaN,NaN
3,2019-01-04,68.0,NaN,NaN,NaN,NaN,NaN
4,2019-01-05,71.0,NaN,NaN,NaN,NaN,NaN
5,2019-01-06,65.0,NaN,NaN,NaN,NaN,NaN
6,2019-01-07,65.0,NaN,NaN,NaN,NaN,NaN
7,2019-01-08,65.0,67.571429,NaN,NaN,2.070197,NaN
8,2019-01-09,69.0,67.142857,NaN,NaN,2.267787,NaN
9,2019-01-10,69.0,67.285714,NaN,NaN,2.360387,NaN


In [82]:
FORECAST_HORIZON = 56

demand_df["target"] = (
    demand_df
    .groupby(["supermarket", "sku"])["demand"]
    .shift(-FORECAST_HORIZON)
)

In [83]:
demand_df[
    [
        "date",
        "demand",
        "target"
    ]
].tail(65)

,date,demand,target
9790,2021-10-27,96.0,95.0
9791,2021-10-28,96.0,98.0
9792,2021-10-29,94.0,95.0
9793,2021-10-30,94.0,95.0
9794,2021-10-31,95.0,96.0
...,...,...,...
9850,2021-12-26,96.0,NaN
9851,2021-12-27,92.0,NaN
9852,2021-12-28,94.0,NaN
9853,2021-12-29,95.0,NaN


In [84]:
final_df = demand_df.dropna().reset_index(drop=True)

In [85]:
print(final_df.shape)

print(final_df.isnull().sum())

(8847, 26)
date               0
demand             0
sku                0
supermarket        0
promotion          0
year               0
month              0
quarter            0
week_of_year       0
day_of_week        0
day_of_month       0
is_weekend         0
lag_1              0
lag_7              0
lag_14             0
lag_28             0
lag_56             0
demand_change_1    0
demand_change_7    0
rolling_mean_7     0
rolling_std_7      0
rolling_mean_14    0
rolling_std_14     0
rolling_mean_28    0
rolling_std_28     0
target             0
dtype: int64


In [86]:
from sklearn.preprocessing import LabelEncoder

sku_encoder = LabelEncoder()
market_encoder = LabelEncoder()

final_df["sku"] = sku_encoder.fit_transform(final_df["sku"])

final_df["supermarket"] = market_encoder.fit_transform(final_df["supermarket"])

In [87]:
import pickle

with open("../models/sku_encoder.pkl", "wb") as f:
    pickle.dump(sku_encoder, f)

with open("../models/supermarket_encoder.pkl", "wb") as f:
    pickle.dump(market_encoder, f)

In [88]:
final_df.head()

,date,demand,sku,supermarket,promotion,year,month,quarter,week_of_year,day_of_week,...,lag_56,demand_change_1,demand_change_7,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14,rolling_mean_28,rolling_std_28,target
0,2019-02-26,66.0,0,0,0,2019,2,1,9,1,...,68.0,0.0,-1.0,67.714286,2.138090,68.071429,1.979288,67.571429,2.201491,71.0
1,2019-02-27,70.0,0,0,0,2019,2,1,9,2,...,68.0,4.0,-1.0,67.571429,2.225395,67.928571,2.055547,67.392857,2.114137,71.0
2,2019-02-28,70.0,0,0,0,2019,2,1,9,3,...,68.0,0.0,0.0,67.428571,1.988060,68.071429,2.129077,67.464286,2.168497,71.0
3,2019-03-01,70.0,0,0,0,2019,3,1,9,4,...,68.0,0.0,2.0,67.428571,1.988060,68.000000,2.038099,67.535714,2.219145,71.0
4,2019-03-02,67.0,0,0,0,2019,3,1,9,5,...,71.0,-3.0,0.0,67.714286,2.214670,68.285714,2.016416,67.571429,2.251396,69.0


In [89]:
final_df.tail()

,date,demand,sku,supermarket,promotion,year,month,quarter,week_of_year,day_of_week,...,lag_56,demand_change_1,demand_change_7,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14,rolling_mean_28,rolling_std_28,target
8842,2021-10-31,95.0,2,2,0,2021,10,4,43,6,...,95.0,1.0,1.0,94.571429,1.133893,94.428571,1.452546,94.607143,1.547741,96.0
8843,2021-11-01,95.0,2,2,0,2021,11,4,44,0,...,95.0,0.0,0.0,94.714286,1.112697,94.571429,1.398586,94.607143,1.547741,92.0
8844,2021-11-02,96.0,2,2,0,2021,11,4,44,1,...,92.0,1.0,3.0,94.714286,1.112697,94.642857,1.392681,94.714286,1.462042,94.0
8845,2021-11-03,92.0,2,2,0,2021,11,4,44,2,...,96.0,-4.0,-4.0,95.142857,0.899735,94.928571,1.206666,94.750000,1.481366,95.0
8846,2021-11-04,95.0,2,2,0,2021,11,4,44,3,...,99.0,3.0,-1.0,94.571429,1.397276,94.642857,1.392681,94.607143,1.547741,96.0


In [90]:
final_df["target"].describe()

count    8847.000000
mean       74.377416
std        23.752792
min        10.000000
25%        57.000000
50%        71.000000
75%        93.000000
max       485.000000
Name: target, dtype: float64

In [91]:
final_df.to_csv(
    "../data/processed/final_dataset.csv",
    index=False
)

In [92]:
FEATURES = [
    col for col in final_df.columns
    if col != "target"
]

TARGET = "target"

print(FEATURES)

['date', 'demand', 'sku', 'supermarket', 'promotion', 'year', 'month', 'quarter', 'week_of_year', 'day_of_week', 'day_of_month', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'lag_56', 'demand_change_1', 'demand_change_7', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_mean_28', 'rolling_std_28']


In [93]:
X = final_df[FEATURES]
y = final_df[TARGET]

# Feature Engineering Summary

The raw demand dataset was transformed into a machine learning-ready dataset by engineering temporal, historical, promotional, and statistical features.

## The following preprocessing steps were performed:

* Missing demand values were imputed using the median demand within each supermarket-SKU combination.
* Promotion periods were converted into a binary promotion indicator.
* Calendar-based features including year, month, quarter, week of year, day of week, day of month, and weekend indicator were created.
* Historical demand information was captured using lag features at 1, 7, 14, 28, and 56-day intervals.
* Rolling mean and rolling standard deviation features were generated using historical demand while avoiding data leakage through shifted rolling windows.
* The forecasting target was created by shifting demand 56 days into the future for each supermarket-SKU combination.
* Categorical variables were label encoded to prepare the dataset for tree-based machine learning models.

The final processed dataset contains 8,847 observations with 24 engineered features and is ready for model training.